In [1]:

import json
import hashlib
import requests
import torch
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import os
import sys
sys.path.append("..")
from utils.balanced_builders_hdf import *
from models.autoencoder_classifier import *
from config import *
import h5py
from captum.attr import IntegratedGradients
from tqdm import tqdm


In [2]:
cs = build_balanced_cs_loaders_from_h5(
    h5_path=xrd_dataset,
    per_class_cs=105000,   
    batch_size=256,
    val_split=0.1,
    test_split=0.1,
    seed=42,
    num_workers=4,
)
print(cs["counts"])
print(cs["sizes"])

{'triclinic': 105000, 'monoclinic': 105000, 'orthorhombic': 105000, 'tetragonal': 105000, 'trigonal': 105000, 'hexagonal': 105000, 'cubic': 105000}
{'train': 588000, 'val': 73500, 'test': 73500}


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = DeepConvAutoencoderClassifier(
    input_length=cs["input_len"],
    latent_dim=64,
    cls_dim=128,
    num_classes=cs["num_classes"],
    use_projection_head=True
).to(device)

checkpoint = torch.load(CS_Cls,
    map_location=device
)

model.load_state_dict(checkpoint)   

model.eval();

C:\Users\doaam\AppData\Local\Temp\ipykernel_24368\2059831275.py:11: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(CS_Cls,


In [ ]:
def get_logits(x):
    _, _, logits, _ = model(x)
    return logits


model.eval()
ig = IntegratedGradients(get_logits)

feature_length = cs["input_len"]
num_classes = cs["num_classes"]


# ===================================================
# Create HDF5 database
# ===================================================
with h5py.File(IG_database_CS, "w") as f:

    d_attrib = f.create_dataset(
        "attributions",
        shape=(0, feature_length),
        maxshape=(None, feature_length),
        dtype="float32",
        compression="gzip"
    )

    d_true = f.create_dataset(
        "true_classes",
        shape=(0,),
        maxshape=(None,),
        dtype="int32",
        compression="gzip"
    )

    d_indices = f.create_dataset(
        "indices",
        shape=(0,),
        maxshape=(None,),
        dtype="int64",
        compression="gzip"
    )

    write_ptr = 0
    global_index = 0

    # ===================================================
    # MAIN LOOP
    # ===================================================
    for x_batch, y_batch in tqdm(cs["train_loader"], desc="Building IG DB"):

        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        batch_size = x_batch.size(0)

        # ---------------------------------------------------
        # IMPORTANT:
        # We DO NOT filter correct predictions.
        # IG should represent CLASS PHYSICS, not accuracy.
        # ---------------------------------------------------

        for i in range(batch_size):

            x = x_batch[i:i+1]
            true_class = int(y_batch[i].item())
            dataset_index = global_index + i

            baseline = torch.zeros_like(x)

            # ---------------------------------------------------
            # TRUE CLASS INTEGRATED GRADIENTS
            # ---------------------------------------------------
            attributions = ig.attribute(
                x,
                baselines=baseline,
                target=true_class,
                n_steps=64,          
                internal_batch_size=16
            )

            # ---------------------------------------------------
            # PHYSICS-AWARE ATTRIBUTION
            # Use magnitude (important for XRD peaks)
            # ---------------------------------------------------
            attrs = np.abs(
                attributions.squeeze().detach().cpu().numpy()
            )

            #  IMPORTANT:
            # DO NOT normalize here.
            # Normalization happens AFTER class averaging.

            # ---------------------------------------------------
            # Resize datasets
            # ---------------------------------------------------
            d_attrib.resize(write_ptr + 1, axis=0)
            d_true.resize(write_ptr + 1, axis=0)
            d_indices.resize(write_ptr + 1, axis=0)

            # ---------------------------------------------------
            # Write data
            # ---------------------------------------------------
            d_attrib[write_ptr] = attrs.astype(np.float32)
            d_true[write_ptr] = true_class
            d_indices[write_ptr] = dataset_index

            write_ptr += 1

            if write_ptr % 5000 == 0:
                print(f"Saved {write_ptr} IG samples...")
                f.flush()

        global_index += batch_size

    print("\n IG database saved successfully!")
    print(f"Total samples stored: {write_ptr}")

/home/users/mohamdmj/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Building IG DB:   1%|          | 19/2296 [04:44<8:09:45, 12.91s/it]

Saved 5000 IG samples...


Building IG DB:   2%|▏         | 39/2296 [09:06<8:13:58, 13.13s/it]

Saved 10000 IG samples...


Building IG DB:   3%|▎         | 58/2296 [13:17<8:11:53, 13.19s/it]

...

Saved 580000 IG samples...


Building IG DB: 100%|█████████▉| 2285/2296 [8:36:46<02:32, 13.88s/it]

Saved 585000 IG samples...


Building IG DB: 100%|██████████| 2296/2296 [8:39:21<00:00, 13.57s/it]


 IG database saved successfully!
Total samples stored: 587776
